## Requirements
- A pre-created vector search endpoint
- Serverless compute version 4
- Required libraries installed to the serverless compute
- Access to Foundational Model APIs for embedding generation
- Permission to create and manage vector search indexes

## Setup

In [0]:
%pip install databricks-vectorsearch==0.60 -q
%restart_python

In [0]:
catalog = "workspace"

# Source table
schema_input = "silver"
table = "docs_chunked"
docs_table = f"{catalog}.{schema_input}.{table}"
embedding_source_column = "chunk"
primary_key = "id"

# Embedding table (index is a special table)
schema_output = "feature_model"
index = "docs_chunked_index"
index_name = f"{catalog}.{schema_output}.{index}"

# Vector search endpoint
VECTOR_SEARCH_ENDPOINT_NAME = "vs_endpoint_1"

# Embedding model endpoint
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

## A. Prepare Source Table
- Vector Search requires the source table, which contains the raw text chunks to be vectorized, to have Change Data Feed (CDF) enabled (one-off effort). If the table already has this feature enabled, we don't need to make any changes.
- Need to have a unique ID for the source table.

In [0]:
# Enable CDF for vector search sync.
spark.sql(f"ALTER TABLE {docs_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Display sample data
display(spark.sql(f"SELECT * FROM {docs_table} LIMIT 5"))

## B. Sample Computing Embeddings for Input Text/Chunk

In [0]:
import mlflow.deployments


# Initialize deployment client to access the embedding model
deploy_client = mlflow.deployments.get_deploy_client("databricks")

# Generate embeddings for a sample text
question = "How Generative AI impacts humans?"
response = deploy_client.predict(
    endpoint=EMBEDDING_MODEL_ENDPOINT_NAME,
    inputs={"input":[question]}
)
embeddings = [e["embedding"] for e in response.data] # list of n-dim embedding

# Display embedding info
print("Embedding for question:", embeddings[0])
print("Embedding dimension:", len(embeddings[0]))

## C. Creating a Vector Search Index
Databricks supports 2 approaches to generate embeddings:
- Managed embeddings: Vector Search automatically computes and manages embeddings, simplifying setup and maintenance. This is the recommended approach for most use cases.
- Manual embeddings: You generate embeddings by your preferred method and store them in a column. For large dataset, consider using UDF.

This demo uses Managed embeddings approach

In [0]:
from databricks.vector_search.client import VectorSearchClient


# init the client and create the index
# this would take ~20min
vsc = VectorSearchClient(disable_notice=True)
vsc.create_delta_sync_index_and_wait(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=index_name,
    source_table_name=docs_table,
    primary_key=primary_key,
    embedding_source_column=embedding_source_column,
    embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
    pipeline_type="TRIGGERED", # for demo, the sync is triggerred only when this code runs.
)
print(f"Index '{index_name}' created for source table '{docs_table}' using endpoint '{VECTOR_SEARCH_ENDPOINT_NAME}'")